# 03 -- IBD Analysis: Florida and Greater Antilles

Computes pairwise site-level FST, linearizes (FST / 1−FST), and regresses against geographic distance.

**Inputs**
- `apal_ibd.tsv` from `01_prepare_samples.ipynb`
- `maf_ld.gt_matrix.int8.npz` from `convert_vcf_to_npz.py`

**Outputs**
- `ibd_florida.pdf / .eps`
- `ibd_greater_antilles.pdf / .eps`

## 1 · Imports and paths

In [ ]:
from __future__ import annotations

import itertools
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, str(Path("../scripts")))
from gt_io import load_cache_npz
from popgen_utils import (
    assign_florida_name_family,
    build_symmetric_pair_matrix,
    compute_ibd_stats_from_long_df,
    dosage_to_genotype_array,
    fst_pair_with_nloci,
    haversine_km,
    linearized_fst,
    order_sites_greedy_nn,
)

In [ ]:
PROJECT_DIR = Path("/Users/kidelu001/awi-work/science/projects/apal_popgen/")
DERIV_DIR   = PROJECT_DIR / "deriv" / "manuscript"
PLOTS_DIR   = PROJECT_DIR / "output" / "plots" / "ibd"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

APAL_IBD_TSV = DERIV_DIR / "sample_registry" / "apal_ibd.tsv"
NPZ_PATH     = DERIV_DIR / "maf_ld.gt_matrix.int8.npz"

## 2 · Parameters

In [ ]:
ID_COL   = "affymetrix_id"
MLG_COL  = "mlg"
SITE_COL = "reef"
LAT_COL  = "lat_clean"
LON_COL  = "lon_clean"
IBD_COL  = "subregion_ibd"

MIN_GENETS_PER_SITE = 5
N_MANTEL_PERM       = 10_000

## 3 · Site merge dictionaries

In [ ]:
# ── Florida ──────────────────────────────────────────────────────────────────
# Name harmonization is handled by assign_florida_name_family() from popgen_utils.
# No manual dict required; all rules are encoded in that function.

# ── Greater Antilles ─────────────────────────────────────────────────────────
GA_SITE_MERGE_MAP = {
    # Dominican Republic
    "Minitas":  "DR_Minitas_Catalina",
    "Catalina": "DR_Minitas_Catalina",

    # Puerto Rico: Laurel / San Cristobal cluster
    "Laurel":                  "PR_Laurel_SanCristobal",
    "San Cristobal":           "PR_Laurel_SanCristobal",
    "San Cristobal Forereef":  "PR_Laurel_SanCristobal",

    # Puerto Rico: Cayo Largo / Palomino cluster
    "Cayo Largo Norte":       "PR_CayoLargo_Palomino",
    "Cayo Largo North":       "PR_CayoLargo_Palomino",
    "Cayo Largo Sur":         "PR_CayoLargo_Palomino",
    "Cayo Largo South":       "PR_CayoLargo_Palomino",
    "Wave Attenuation Site":  "PR_CayoLargo_Palomino",
    "Lobos":                  "PR_CayoLargo_Palomino",
    "Palomino (Sand Slide )": "PR_CayoLargo_Palomino",

    # Puerto Rico: individual consolidated sites
    "Gilligan's Reef":  "PR_Gilligans",
    "Cayo Lajas":       "PR_CayoLajas",
    "Coral Uberbs":     "PR_CayoCoral_Uberbs",
    "Cayo Coral HMO1":  "PR_CayoCoral_Uberbs",
    "Cayo Coral HMO4":  "PR_CayoCoral_Uberbs",
    "Cayo Coral HMO5":  "PR_CayoCoral_Uberbs",
    "Cayo Coral HMO9":  "PR_CayoCoral_Uberbs",
}

## 4 · Load data

In [ ]:
apal_ibd = pd.read_csv(APAL_IBD_TSV, sep="\t", low_memory=False)
gt = load_cache_npz(str(NPZ_PATH))

print(f"Metadata : {len(apal_ibd):,} samples")
print(f"NPZ      : {gt.G.shape[0]:,} SNPs × {gt.G.shape[1]:,} samples")

apal_ibd[IBD_COL].value_counts()

## 5 · Helper functions

In [ ]:
def build_panel(apal_ibd, gt, subregion, site_merge_fn=None, site_merge_map=None):
    """
    Filter to subregion, apply site merges, align to NPZ order,
    deduplicate (one sample per MLG × site), filter by min genets,
    and return a trimmed GenotypeArray.

    Parameters
    ----------
    site_merge_fn  : callable  – e.g. assign_florida_name_family
    site_merge_map : dict      – e.g. GA_SITE_MERGE_MAP

    Returns
    -------
    panel       : pd.DataFrame  sample metadata aligned to gta columns
    site_coords : pd.DataFrame  per-site centroid + n_genets
    gta         : allel.GenotypeArray
    """
    panel = apal_ibd[apal_ibd[IBD_COL] == subregion].copy()

    if site_merge_fn is not None:
        panel["site_merged"] = panel[SITE_COL].map(site_merge_fn)
    elif site_merge_map is not None:
        panel["site_merged"] = panel[SITE_COL].map(lambda x: site_merge_map.get(x, x))
    else:
        panel["site_merged"] = panel[SITE_COL]

    # Align to NPZ sample order
    npz_lookup = {s: i for i, s in enumerate(gt.sample_ids)}
    panel = panel[panel[ID_COL].isin(npz_lookup)].copy()
    panel["_npz_idx"] = panel[ID_COL].map(npz_lookup)
    panel = panel.sort_values("_npz_idx").reset_index(drop=True)

    # Deduplicate: one sample per (MLG, merged site)
    panel = (
        panel
        .drop_duplicates(subset=[MLG_COL, "site_merged"], keep="first")
        .reset_index(drop=True)
    )

    # Filter sites by minimum genets
    site_n = panel.groupby("site_merged")[MLG_COL].nunique()
    valid  = site_n[site_n >= MIN_GENETS_PER_SITE].index
    panel  = panel[panel["site_merged"].isin(valid)].copy().reset_index(drop=True)

    # Site centroids and genet counts
    site_coords = (
        panel.groupby("site_merged")
        .agg(
            lat_median=(LAT_COL, "median"),
            lon_median=(LON_COL, "median"),
            n_genets=(MLG_COL, "nunique"),
        )
        .reset_index()
        .rename(columns={"site_merged": "site"})
    )

    # Subset genotype matrix and build GenotypeArray
    G_sub = gt.G[:, panel["_npz_idx"].to_numpy()]
    gta   = dosage_to_genotype_array(G_sub)

    return panel, site_coords, gta


def compute_pairwise_fst(panel, gta):
    """Weir-Cockerham FST for all site pairs. Returns a long DataFrame."""
    sites = sorted(panel["site_merged"].unique())
    rows  = []
    for s1, s2 in itertools.combinations(sites, 2):
        mask1 = (panel["site_merged"] == s1).values
        mask2 = (panel["site_merged"] == s2).values
        fst, n_loci = fst_pair_with_nloci(gta, mask1, mask2)
        rows.append({"site1": s1, "site2": s2, "fst": fst, "n_loci": n_loci})
    return pd.DataFrame(rows)


def build_ibd_df(fst_df, site_coords_ordered):
    """Add linearized FST, geographic distance, and along-tract distance."""
    df       = fst_df.copy()
    df["linearized_fst"] = df["fst"].apply(linearized_fst)
    coord    = site_coords_ordered.set_index("site")[["lat_median", "lon_median", "tract_km"]]

    df["geo_dist_km"]    = df.apply(lambda r: haversine_km(
        coord.loc[r["site1"], "lat_median"], coord.loc[r["site1"], "lon_median"],
        coord.loc[r["site2"], "lat_median"], coord.loc[r["site2"], "lon_median"],
    ), axis=1)
    df["along_tract_km"] = df.apply(lambda r: abs(
        coord.loc[r["site1"], "tract_km"] - coord.loc[r["site2"], "tract_km"]
    ), axis=1)
    return df


def plot_ibd(ibd_df, x_col="geo_dist_km", y_col="linearized_fst",
             title="IBD", save_base=None, figsize=(5, 5)):
    """IBD scatter with OLS regression line and annotated statistics."""
    df   = ibd_df[[x_col, y_col]].dropna()
    x, y = df[x_col].values, df[y_col].values

    ols  = stats.linregress(x, y)
    r2   = ols.rvalue ** 2
    istats = compute_ibd_stats_from_long_df(
        ibd_df, x_col=x_col, y_col=y_col, n_perm=N_MANTEL_PERM
    )

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(x, y, s=40, edgecolors="k", linewidths=0.5, alpha=0.8, zorder=3)

    xline = np.linspace(x.min(), x.max(), 200)
    ax.plot(xline, ols.slope * xline + ols.intercept, color="0.4", lw=1)

    label = (
        f"$R^2$ = {r2:.3f}  (p = {istats['ols_p']:.3f})\n"
        f"Mantel r = {istats['mantel_r']:.3f}  (p = {istats['mantel_p']:.3f})\n"
        f"n pairs = {istats['n_pairs']}"
    )
    ax.text(0.97, 0.05, label, transform=ax.transAxes, fontsize=8,
            ha="right", va="bottom",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.8", alpha=0.9))

    ax.set_xlabel("Geographic distance (km)")
    ax.set_ylabel("$F_{ST}$ / (1 − $F_{ST}$)")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()

    if save_base is not None:
        for ext in ("pdf", "eps"):
            fig.savefig(f"{save_base}.{ext}", bbox_inches="tight")
        print(f"Saved → {save_base}.pdf / .eps")

    plt.show()
    return istats

---
## Florida panel

In [ ]:
fl_panel, fl_sites, fl_gta = build_panel(
    apal_ibd, gt,
    subregion="Florida",
    site_merge_fn=assign_florida_name_family,
)

print(f"{fl_panel['site_merged'].nunique()} sites | {fl_panel[MLG_COL].nunique()} genets")
fl_sites.sort_values("lon_median")

In [ ]:
fl_fst_df = compute_pairwise_fst(fl_panel, fl_gta)
print(f"{len(fl_fst_df)} site pairs | FST range: "
      f"{fl_fst_df['fst'].min():.4f} – {fl_fst_df['fst'].max():.4f}")
fl_fst_df.head()

In [ ]:
fl_sites_ordered = order_sites_greedy_nn(fl_sites)
fl_ibd_df = build_ibd_df(fl_fst_df, fl_sites_ordered)
fl_ibd_df.head()

In [ ]:
fl_stats = plot_ibd(
    fl_ibd_df,
    x_col="geo_dist_km",
    title="IBD — Florida",
    save_base=PLOTS_DIR / "ibd_florida",
)
fl_stats

---
## Greater Antilles panel

In [ ]:
ga_panel, ga_sites, ga_gta = build_panel(
    apal_ibd, gt,
    subregion="GreaterAntilles",
    site_merge_map=GA_SITE_MERGE_MAP,
)

print(f"{ga_panel['site_merged'].nunique()} sites | {ga_panel[MLG_COL].nunique()} genets")
ga_sites.sort_values("lon_median")

In [ ]:
ga_fst_df = compute_pairwise_fst(ga_panel, ga_gta)
print(f"{len(ga_fst_df)} site pairs | FST range: "
      f"{ga_fst_df['fst'].min():.4f} – {ga_fst_df['fst'].max():.4f}")
ga_fst_df.head()

In [ ]:
ga_sites_ordered = order_sites_greedy_nn(ga_sites)
ga_ibd_df = build_ibd_df(ga_fst_df, ga_sites_ordered)
ga_ibd_df.head()

In [ ]:
ga_stats = plot_ibd(
    ga_ibd_df,
    x_col="geo_dist_km",
    title="IBD — Greater Antilles",
    save_base=PLOTS_DIR / "ibd_greater_antilles",
)
ga_stats